# Module 2 — Sentiment / Emotion Classifier

**Requirement:** Build a multi-class classifier using either Recurrent
Neural Networks or Transformers to classify the emotional tone of the
customer's message (frustrated/negative, neutral, satisfied/positive).

**Dataset:** `dair-ai/emotion` — 20k English Twitter messages labeled with
6 emotions (sadness, joy, love, anger, fear, surprise).

**Approach used here:** RNN (LSTM), one of the two options allowed by the
requirement.


In [1]:
# 1. Import libraries
from datasets import load_dataset
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, classification_report

c:\Users\i9\Downloads\chatbot-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the dataset

In [2]:
dataset = load_dataset("dair-ai/emotion")

train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

label_names = dataset["train"].features["label"].names
print(label_names)
print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()

c:\Users\i9\Downloads\chatbot-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\i9\.cache\huggingface\hub\datasets--dair-ai--emotion. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\i9\Downloads\chatbot-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarnin

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
(16000, 2) (2000, 2) (2000, 2)


,text,label
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,3
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,3


## 3. Tokenize and pad the text

In [3]:
VOCAB_SIZE = 10000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

X_train = pad_sequences(tokenizer.texts_to_sequences(train_df["text"]), maxlen=MAX_LEN)
X_val = pad_sequences(tokenizer.texts_to_sequences(val_df["text"]), maxlen=MAX_LEN)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_df["text"]), maxlen=MAX_LEN)

num_classes = len(label_names)
y_train = to_categorical(train_df["label"], num_classes=num_classes)
y_val = to_categorical(val_df["label"], num_classes=num_classes)
y_test_labels = test_df["label"].values

## 4. Build the RNN (LSTM) model

In [4]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=64, input_length=MAX_LEN),
    LSTM(64),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

c:\Users\i9\Downloads\chatbot-project\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 5. Train the model

In [5]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64
)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 45s 145ms/step - accuracy: 0.4574 - loss: 1.3887 - val_accuracy: 0.6635 - val_loss: 0.9474
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 34s 120ms/step - accuracy: 0.8113 - loss: 0.5670 - val_accuracy: 0.8755 - val_loss: 0.3773
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 30s 121ms/step - accuracy: 0.9344 - loss: 0.1967 - val_accuracy: 0.8865 - val_loss: 0.3326
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 31s 122ms/step - accuracy: 0.9622 - loss: 0.1131 - val_accuracy: 0.8915 - val_loss: 0.3334
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 30s 119ms/step - accuracy: 0.9735 - loss: 0.0776 - val_accuracy: 0.9020 - val_loss: 0.2997


## 6. Evaluate on the test set

In [6]:
test_probs = model.predict(X_test)
test_preds = np.argmax(test_probs, axis=1)

print("Test accuracy:", accuracy_score(y_test_labels, test_preds))
print(classification_report(y_test_labels, test_preds, target_names=label_names))

63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step
Test accuracy: 0.903
              precision    recall  f1-score   support

     sadness       0.95      0.96      0.95       581
         joy       0.92      0.93      0.93       695
        love       0.76      0.74      0.75       159
       anger       0.91      0.87      0.89       275
        fear       0.85      0.90      0.87       224
    surprise       0.76      0.64      0.69        66

    accuracy                           0.90      2000
   macro avg       0.86      0.84      0.85      2000
weighted avg       0.90      0.90      0.90      2000



## 7. Save the model and tokenizer (for deployment)

In [7]:
import pickle

model.save("sentiment_emotion_model.h5")

with open("sentiment_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)